# Build a ReAct Agent with LangGraph on Flyte

<a target="_blank" href="https://colab.research.google.com/github/unionai/workshops/blob/main/tutorials/starter-examples/langgraph-react-agent/tutorial_langgraph_react_agent.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

Build a simple ReAct (Reason + Act) agent using LangGraph's prebuilt `create_react_agent` and run it on Flyte.

**What you'll learn:**
- Define tools with `@tool` and `@flyte.trace` for observability
- Use LangGraph's prebuilt ReAct agent
- Run the agent as a Flyte task locally and remotely

**Pattern:**
```
User: "What is 12 * 7 plus 3?"
  → Agent reasons: need to multiply first
  → Calls multiply(12, 7) → 84
  → Reasons: now add 3
  → Calls add(84, 3) → 87
  → Returns: "87"
```

---

## Setup

In [1]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !git clone https://github.com/unionai/workshops
    %cd workshops/tutorials/starter-examples/langgraph-react-agent/
    !pip install -r requirements.txt

from utils.file_viewer import view_file

## Dependencies

The example uses LangGraph, LangChain, and Flyte:

In [2]:
view_file("requirements.txt")

## Set your API Key

The agent uses OpenAI, so you'll need an API key.

You can set it as an environment variable, in a `.env`, or enter it below:

In [9]:
import os
from getpass import getpass

if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass('OPENAI_API_KEY: ')

To run on a remote Flyte cluster, add the API key as a secret:

```bash
flyte create secret OPENAI_API_KEY
```

## Connect to Flyte Cluster

Skip this step if you only want to run locally.

- Don't have a Flyte cluster? Request access at [flyte.org](https://flyte.org/)
- Already have one? Set your endpoint below:

In [5]:
!flyte create config \
    --endpoint <your-endpoint> \
    --auth-type headless \
    --builder remote \
    --domain development \
    --project flytesnacks

zsh:1: no such file or directory: your-endpoint


---

## Code Walkthrough

The entire agent fits in a single file. Let's walk through it:

In [3]:
view_file("langgraph_react_agent.py")

### Key Components

**1. Flyte Environment** — Defines the container image, secrets, and resources for the task.

```python
env = flyte.TaskEnvironment(
    name="langgraph_env",
    image=flyte.Image.from_debian_base().with_requirements("requirements.txt"),
    secrets=[flyte.Secret(key="OPENAI_API_KEY", as_env_var="OPENAI_API_KEY")],
)
```

**2. Tools** — Simple Python functions decorated with `@tool` (LangChain) and `@flyte.trace` (observability). Must be `async`.

```python
@tool
@flyte.trace
async def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b
```

**3. Agent Task** — Uses LangGraph's `create_react_agent` to wire up the LLM + tools into a ReAct loop, wrapped as a Flyte task.

```python
@env.task
async def agent(request: str) -> str:
    llm = ChatOpenAI(model="gpt-4o-mini")
    react_agent = create_react_agent(llm, tools)
    result = await react_agent.ainvoke(...)
```

---

## Run the Agent

### Local run

In [4]:
!flyte run --local langgraph_react_agent.py agent --request "What is 12 * 7 plus 3?"

⠴ Launching local execution...0m
╭────────────────────────────── Local Success ───────────────────────────────╮
│ Completed Local Run                                                        │
│ Path: /tmp/flyte/metadata/ae59caab-f993-4e33-9d0c-eb3f5854490f             │
│ ➡️ Outputs: ActionOutputs(o0="The result of \( 12 \times 7 + 3 \) is 87.") │
╰────────────────────────────────────────────────────────────────────────────╯


### Remote run

The first run will build and push a container image, which may take a few minutes.

In [6]:
!flyte run langgraph_react_agent.py agent --request "What is 12 * 7 plus 3?"

⠧ Launching remote execution...17:02:05.452724 WARNING  remote_builder.py:102 -  Image                         
                         356633062068.dkr.ecr.us-east-2.amazonaws.com/union/demo
                         :flyte-b7bc6980716936a4110ec86d49042651 found. Skip    
                         building.                                              
17:02:05.454988 WARNING  _deploy.py:402 -  Built Image for environment          
                         langgraph_env, image:                                  
                         356633062068.dkr.ecr.us-east-2.amazonaws.com/union/demo
                         :flyte-b7bc6980716936a4110ec86d49042651                
⠙ Launching remote execution...
╭──────────────────────────────────────────────── Remote Run ────────────────────────────────────────────────╮
│ Created Run: rds72qsnlq8wcn8b77vs                                                                          │
│ URL: ]8;id=436012;https://demo.hosted.unionai.cloud/v2/domain/de

### Fetch remote run output

After a remote run completes, you can fetch the output programmatically using `flyte.remote`:

```python
from flyte.remote import Run

# Get a run by name
run = Run.get("your-run-name")
outputs = run.outputs()
```

Replace the run name below with the one printed from the remote run above (e.g. `rvgkvvdwpwqffm2bzh55`):

In [ ]:
import flyte
from flyte.remote import Run

flyte.init_from_config()

# Fetch the latest run of this agent
runs = list(Run.listall(task_name="langgraph_env.agent", sort_by=("created_at", "desc"), limit=1))
run = Run.get(runs[0].name)
print(f"Run: {run.name}")
print(f"Phase: {run.phase}")
if run.phase.name == "SUCCEEDED":
    print(f"Output: {run.outputs()}")
else:
    print("Run has not completed yet.")

### Try different prompts

In [ ]:
!flyte run --local langgraph_react_agent.py agent --request "Multiply 15 by 4 and then add 20"

In [ ]:
!lyte run --local langgraph_react_agent.py agent --request "What is 100 divided by 4 times 3?"

### Local run with TUI

Flyte includes an interactive terminal dashboard (TUI) that shows each task's status, logs, and outputs in real time. Run this from your terminal (not the notebook):

```bash
uv pip install textual

uv run flyte run --local --tui langgraph_react_agent.py agent --request "What is 12 * 7 plus 3?"
```

You can also browse past local runs with:

```bash
uv run flyte start tui
```

---

## Key Takeaways

- **LangGraph + Flyte** — Use LangGraph for agent logic, Flyte for orchestration, scaling, and observability
- **`@flyte.trace`** — Gives you visibility into each tool call in the Flyte dashboard
- **`create_react_agent`** — LangGraph's prebuilt ReAct loop handles the reasoning cycle for you
- **Single file** — The entire agent is self-contained in one `.py` file

## Next Steps

- Add more tools (web search, file I/O, database queries)
- Try different LLMs (`gpt-4o`, `claude-sonnet-4-5-20250929`)
- Add a Flyte report to visualize the agent's reasoning trace
- Check out the [multi-agent tutorials](../../multi-agent-workflows/) for more complex patterns